# Week 3: Official x402 and AP2

**120-minute lab.** First complete a standalone 30-minute x402 experiment: demonstration 5 min, startup 5 min, two short coding tasks 15 min, verification 5 min. Then spend 65 min on hotel AP2 (walkthrough 15, two tasks 30, verification 20), with 25 min for discussion and buffer.

Exactly four tasks: two in `src/travel_challenge/week3/paid_weather.py` and two in `src/travel_challenge/week3/authorization.py`. No additional tasks or payment infrastructure.

Weather flow: HTTP request -> 402 -> inspect requirements -> official x402 SDK -> retry -> weather data. Use fixed Tokyo. No travel plan, Nuitee key, A2A, MCP or AP2 is needed for this experiment; weather does not drive any later purchase.

Only the separate AP2 section needs your exported hotel plan. Real SDK operations and signatures are used, but all settlement and hotel execution are simulated. NO funds move.

## Standalone x402 Setup

Teacher setup installs the pinned official x402 SDK and its EVM dependencies. Use the prepared Python environment. No private key entry, wallet funding, RPC or facilitator configuration is needed.

Run only the weather section through its shutdown cell for the independent experiment. It uses Tokyo and does not read `trip_plan.json`, import AP2 or require earlier student answers.

Keep startup code instructor-provided. If a student task fails, edit the Python function and rerun that task's cell: the Weather API stays running. Rerunning startup safely reuses this notebook's own service; unrelated processes are never stopped.

In [ ]:
import sys
import importlib
import subprocess
from pathlib import Path
from importlib.metadata import version
from travel_challenge.week3 import paid_weather

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
city = 'Tokyo'
print('Official x402:', version('x402'), 'City:', city)
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week3.py', '-m', 'scaffold', '-k', 'x402 or weather', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, check.stderr

## Teacher Weather Service

The teacher provides the complete FastAPI route, official x402 requirements, local EIP-712 verification and simulated receipt. The supplied startup and shutdown cells manage only this notebook's process.

The forecast is a fixed teaching response. The payment challenge quotes 3000 USDC minor units, or 0.003 USDC, using a Base Sepolia signature domain. No on-chain call occurs; no balance or transfer is checked. The response explicitly says `SIMULATED_NO_FUNDS_MOVED`.

In [ ]:
import asyncio
import socket
import tempfile
import time
import httpx

weather_url = 'http://127.0.0.1:8020/weather'
reuse = globals().get('weather_process') is not None and weather_process.poll() is None
if not reuse:
    if globals().get('weather_log') is not None and not weather_log.closed:
        weather_log.close()
    with socket.socket() as sock:
        assert sock.connect_ex(('127.0.0.1', 8020)) != 0, 'Port 8020 is occupied by another process; stop its owner first.'
    weather_log = tempfile.TemporaryFile(mode='w+')
    weather_process = subprocess.Popen([sys.executable, '-m', 'travel_challenge.week3.paid_weather'], cwd=ROOT, stdout=weather_log, stderr=subprocess.STDOUT)
try:
    deadline = time.monotonic() + 30
    async with httpx.AsyncClient(timeout=2) as http:
        while True:
            if weather_process.poll() is not None:
                raise RuntimeError('Weather service exited during startup.')
            try:
                probe = await http.get(weather_url, params={'city': 'Tokyo'})
                if probe.status_code == 402 and probe.json().get('settlement') == 'SIMULATED_NO_FUNDS_MOVED':
                    break
            except httpx.TransportError:
                pass
            if time.monotonic() > deadline:
                raise RuntimeError('Weather startup exceeded 30 seconds.')
            await asyncio.sleep(0.1)
    print('Weather API reused.' if reuse else 'Weather API started.', 'Unpaid request: HTTP 402.')
except Exception:
    weather_process.terminate()
    try:
        weather_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        weather_process.kill()
        weather_process.wait()
    weather_log.seek(0)
    print(weather_log.read()[-4000:])
    weather_log.close()
    weather_process = None
    raise

## Task 1: Inspect HTTP 402 (5 min)

Edit `src/travel_challenge/week3/paid_weather.py::read_requirements`. Require status 402 and call the official `decode_payment_required_header` on `PAYMENT-REQUIRED`. Return the official `PaymentRequired` object.

Run the next cell and inspect resource, scheme, network, asset, recipient and amount. This is USDC minor units, not hotel SGD cents. No custom Base64 or JSON protocol parser.

If the TODO raises, leave the server running. Edit the function and rerun this cell; the module reloads your changes.

In [ ]:
importlib.reload(paid_weather)
async with httpx.AsyncClient() as http:
    unpaid = await http.get(weather_url, params={'city': 'Tokyo'})
print('Actual HTTP status:', unpaid.status_code)
required = paid_weather.read_requirements(unpaid)
print('Official object:', type(required).__module__, type(required).__name__)
print('Resource:', required.resource)
offer = required.accepts[0]
print({'scheme': offer.scheme, 'network': offer.network, 'asset': offer.asset, 'recipient': offer.pay_to, 'amount': offer.amount})

## Task 2: Official SDK Signing and Retry (10 min)

Edit `src/travel_challenge/week3/paid_weather.py::pay_for_weather`. Requirements decoding and a disposable unfunded SDK client are provided.

Complete this short sequence: `await client.create_payment_payload(required)` -> `encode_payment_signature_header(...)` -> one `http.get(response.request.url, headers=...)` -> require success -> `decode_payment_response_header(...)`. Return `(weather_dictionary, official_settlement_response)`. Imported official functions are already available in the source file.

Do not implement cryptography, fund a wallet or configure RPC. The next cell verifies only this standalone weather exercise. After completion, run the request cells again to repeat the experiment; no service restart is needed.

In [ ]:
importlib.reload(paid_weather)
async with httpx.AsyncClient(timeout=30) as http:
    unpaid = await http.get(weather_url, params={'city': 'Tokyo'})
    forecast, settlement = await paid_weather.pay_for_weather(http, unpaid)
assert forecast['city'] == 'Tokyo'
assert forecast['settlement'] == 'SIMULATED_NO_FUNDS_MOVED'
assert settlement.success and settlement.transaction.startswith('SIMULATED:')
print(forecast)
print('Official simulated receipt:', settlement.model_dump(by_alias=True))
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week3.py', '-k', 'x402 or weather', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, 'Complete the two weather TODOs. ' + check.stderr

## What Was Real, What Was Simulated?

Real: HTTP 402, official x402 schemas/header functions, the official exact-EVM client, EIP-712 signing and local signature recovery.

Simulated: the forecast, settlement success and transaction identifier. No RPC, token balance, on-chain nonce, facilitator verification or funds transfer was checked. A successful simulated receipt is not evidence of a real USDC payment.

The signing account is disposable and unfunded. Do not fund it or send these teaching payloads to a real settlement service. Shut down the local API below before moving to hotel authorization.

In [ ]:
if globals().get('weather_process') is not None:
    failed = weather_process.poll() not in [None, 0]
    weather_process.terminate()
    try:
        weather_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        weather_process.kill()
        weather_process.wait()
    if not weather_log.closed:
        if failed:
            weather_log.seek(0)
            print(weather_log.read()[-4000:])
        weather_log.close()
    weather_process = None
print('Weather service stopped. Safe to run this shutdown cell again; no funds moved.')

## Hotel AP2: Models and Local Authorization

The purchase comes from the selected hotel in `trip_plan.json`: supplier offer ID, merchant ID/name, full-stay amount in integer cents, currency, rooms and dates. No flight or hotel price is invented here.

The official v0.2.0 Checkout Mandate contains a merchant checkout JWT and its hash. The Payment Mandate binds its transaction ID to that hash and carries payee, amount/currency and instrument.

Teacher helpers use official AP2 `create_jwt`/`verify_jwt` with a local ephemeral merchant key. This verifies that local merchant JWT only. There is no user-signed payment credential or full AP2 trust-chain verification.

## Task 3: Official Checkout/Payment Models (15 min)

Edit `src/travel_challenge/week3/authorization.py::make_mandates`. The teacher supplies `token` and `binding` using the real SDK JWT helper in `sdk_support/ap2_signing.py`. Return official `CheckoutMandate` and `PaymentMandate` instances using the exact field guidance in the function docstring.

The next cell reads the exported `trip_plan.json` for this AP2 section only. Complete the Week 1 or Week 2 export first. If you change `config/trip_scenario.py`, regenerate the plan through W1/W2; this section deliberately uses the saved request, budget and purchase, not today's scenario settings.

The full saved trip must be within its budget. The payment mandate amount is only the selected hotel's exact integer-cent amount, not the trip total or budget. Keep its currency, merchant and the supplied simulated instrument. Do not implement signing or replacement models.

The cell displays essential official fields and checks model types without printing a lengthy JWT. The earlier standalone x402 section does not use this plan or the travel scenario.

In [ ]:
import json
from travel_challenge.week3 import authorization

plan_path = ROOT / 'trip_plan.json'
assert plan_path.exists(), 'AP2 only: run the plan-export cell in Week 1 or Week 2 first.'
plan = json.loads(plan_path.read_text(encoding='utf-8'))
purchase = authorization.hotel_purchase(plan)
print('AP2 source: Git v0.2.0; distribution version:', version('ap2'))
print('Hotel purchase:', purchase)
importlib.reload(authorization)
checkout, payment = authorization.make_mandates(purchase)
assert isinstance(checkout, authorization.CheckoutMandate)
assert isinstance(payment, authorization.PaymentMandate)
print('Checkout hash:', checkout.checkout_hash)
print('Payment binding:', payment.transaction_id)
print('Payee:', payment.payee.model_dump())
print('Amount:', payment.payment_amount.model_dump())
print('Instrument:', payment.payment_instrument.model_dump())

## Task 4: Check Changed Authorization Conditions (15 min)

Edit `src/travel_challenge/week3/authorization.py::check_authorization`. Compare the verified checkout purchase with the current purchase. Check payment binding, merchant, amount, currency and the supplied simulated instrument. Return `False` for changed conditions or invalid checkout verification. Ordinary `if` statements and a small `try/except ValueError` are sufficient; no long Boolean expression is required.

The imported teacher helper `verified_checkout` performs the local JWT/hash check in `sdk_support/ap2_signing.py`. Your comparisons are business authorization, not cryptography or just Pydantic structure validation.

The next cell reloads your authorization code, rebuilds the sample with the current local merchant key, and demonstrates unchanged and changed conditions. The ephemeral key lives in the instructor support module for this process only.

In [ ]:
importlib.reload(authorization)
checkout, payment = authorization.make_mandates(purchase)
assert authorization.check_authorization(purchase, checkout, payment) is True
changed_amount = {**purchase, 'amount_cents': purchase['amount_cents'] + 5000}
changed_merchant = {**purchase, 'merchant_id': 'different-hotel'}
assert authorization.check_authorization(changed_amount, checkout, payment) is False
assert authorization.check_authorization(changed_merchant, checkout, payment) is False
changed_payment = payment.model_copy(deep=True)
changed_payment.payment_amount.amount += 1
assert authorization.check_authorization(purchase, checkout, changed_payment) is False
print('Original accepted; changed amount, merchant and payment rejected.')

## Structure Validation Is Not Authorization

An official Payment Mandate with a different but well-formed amount or merchant can pass Pydantic model validation. It must still fail your local authorization check against the selected hotel purchase.

The teacher can verify the local merchant checkout JWT, but the payment mandate is not user-signed and no full AP2 trust chain is implemented. The approval snapshot below is trusted in-process classroom data, not a portable security credential.

Running the approval call with `consent=True` represents your explicit classroom approval of the displayed purchase. It does not move funds.

In [ ]:
try:
    authorization.approve_purchase(purchase, checkout, payment, consent=False)
except PermissionError:
    print('Without consent: authorization rejected.')
else:
    raise AssertionError('Missing consent was accepted')
print('Explicitly approving this simulated hotel purchase:', purchase)
authorized_purchase = authorization.approve_purchase(purchase, checkout, payment, consent=True)
assert authorized_purchase == purchase and authorized_purchase is not purchase
print('Detached approved snapshot:', authorized_purchase)

## Verify Your Four Tasks

Teacher infrastructure: `python -m pytest tests/test_week3.py -m scaffold -q`. Student acceptance: replace `scaffold` with `student`. All Week 3 checks: omit `-m`.

The tests exercise official x402 signing/header parsing and actual HTTP 402/200 responses, malformed/tampered signature rejection, official AP2 models and local JWT checks, plus your unchanged-purchase authorization rules.

Only student TODO checks should fail before implementation. There are no skip/xfail shortcuts. The next cell runs the complete Week 3 file from the repository root.

In [ ]:
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week3.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, 'Complete the four Week 3 source tasks. ' + check.stderr

## Review and Handoff (25 min Including Walkthrough Discussion)

Identify what an HTTP 402 challenge communicates, what the real x402 signature proves locally, and what the simulated receipt does not prove. Distinguish official AP2 model validation, local merchant JWT verification and business authorization.

Show that a changed hotel amount or merchant is rejected. Week 4 reuses the same purchase and authorization functions, with a small receipt dictionary and two fixed execution feedback scenarios.

Keep `trip_plan.json` from Weeks 1-2. No additional data, wallet or authorization file is needed; Week 4 creates a fresh local approval using the same Week 3 functions.